# PFNRBicop Sinkhorn Projection Check (Unconditional)

This notebook compares `PFNRBicop(method="criterion")` with and without Sinkhorn projection on data simulated from a Clayton copula.

Goal: check how well each model satisfies the copula marginal constraints\n
\(\int_0^1 c(u,v)\,dv = 1\) and \(\int_0^1 c(u,v)\,du = 1\).

In [6]:
import numpy as np
import pandas as pd
import pyvinecopulib as pv
from scipy.stats import kendalltau

from npcc import PFNRBicop

## 1) Simulate training data from a Clayton copula

In [7]:
rho = 3.0
n_train = 1000

clayton = pv.Bicop(
  family=pv.BicopFamily.clayton,
  parameters=np.asarray([[rho]], dtype=np.float64),
)
u_train = clayton.simulate(n=n_train, seeds=[2, 2, 4])

u = u_train[:, 0]
v = u_train[:, 1]

print(f"Generated samples: {u_train.shape}")
print(f"Empirical Kendall tau: {kendalltau(u, v).statistic:.4f}")

Generated samples: (1000, 2)
Empirical Kendall tau: 0.6158


## 2) Fit PFNRBicop with and without Sinkhorn projection (criterion method)

In [11]:
pfnr_no_proj = PFNRBicop(method="criterion", sinkhorn_iters=None)
pfnr_proj = PFNRBicop(method="criterion", sinkhorn_iters=5)
pfnr_proj_identity = PFNRBicop(
  method="criterion",
  sinkhorn_iters=5,
  transform="identity",
)

pfnr_no_proj.fit(u, v)
pfnr_proj.fit(u, v)
pfnr_proj_identity.fit(u, v)

print("Fitted all three models.")

Fitted all three models.


## 3) Evaluate density on a grid and check marginal constraints

In [12]:
grid_n = 201
u_grid = np.linspace(0.01, 0.99, grid_n)
v_grid = np.linspace(0.01, 0.99, grid_n)

c_no_proj = pfnr_no_proj.pdf_grid(u_grid, v_grid)
c_proj = pfnr_proj.pdf_grid(u_grid, v_grid)
c_proj_identity = pfnr_proj_identity.pdf_grid(u_grid, v_grid)


def marginal_diagnostics(
  c: np.ndarray, u_grid: np.ndarray, v_grid: np.ndarray
) -> dict[str, float]:
  # Integrate over v for each fixed u.
  int_over_v = np.trapezoid(c, x=v_grid, axis=1)
  # Integrate over u for each fixed v.
  int_over_u = np.trapezoid(c, x=u_grid, axis=0)

  err_v = np.abs(int_over_v - 1.0)
  err_u = np.abs(int_over_u - 1.0)

  return {
    "mean_abs_err_int_c_dv": float(err_v.mean()),
    "max_abs_err_int_c_dv": float(err_v.max()),
    "mean_abs_err_int_c_du": float(err_u.mean()),
    "max_abs_err_int_c_du": float(err_u.max()),
  }


diag_no_proj = marginal_diagnostics(c_no_proj, u_grid, v_grid)
diag_proj = marginal_diagnostics(c_proj, u_grid, v_grid)
diag_proj_identity = marginal_diagnostics(c_proj_identity, u_grid, v_grid)

report = pd.DataFrame(
  [
    {"model": "criterion / no projection / transform=logit", **diag_no_proj},
    {"model": "criterion / Sinkhorn (5 iters) / transform=logit", **diag_proj},
    {
      "model": "criterion / Sinkhorn (5 iters) / transform=identity",
      **diag_proj_identity,
    },
  ]
)

report

,model,mean_abs_err_int_c_dv,max_abs_err_int_c_dv,mean_abs_err_int_c_du,max_abs_err_int_c_du
0,criterion / no projection / transform=logit,0.027227,0.439878,2.670507e-02,4.079397e-01
1,criterion / Sinkhorn (5 iters) / transform=logit,0.003055,0.026600,2.645755e-16,9.992007e-16
2,criterion / Sinkhorn (5 iters) / transform=ide...,0.004509,0.033713,2.397198e-16,1.332268e-15


## 4) Optional: inspect first few marginal integrals

In [13]:
int_v_no_proj = np.trapezoid(c_no_proj, x=v_grid, axis=1)
int_u_no_proj = np.trapezoid(c_no_proj, x=u_grid, axis=0)
int_v_proj = np.trapezoid(c_proj, x=v_grid, axis=1)
int_u_proj = np.trapezoid(c_proj, x=u_grid, axis=0)
int_v_proj_identity = np.trapezoid(c_proj_identity, x=v_grid, axis=1)
int_u_proj_identity = np.trapezoid(c_proj_identity, x=u_grid, axis=0)

preview = pd.DataFrame(
  {
    "u_idx": np.arange(10),
    "int_c_dv_no_proj": int_v_no_proj[:10],
    "int_c_dv_proj_logit": int_v_proj[:10],
    "int_c_dv_proj_identity": int_v_proj_identity[:10],
    "int_c_du_no_proj": int_u_no_proj[:10],
    "int_c_du_proj_logit": int_u_proj[:10],
    "int_c_du_proj_identity": int_u_proj_identity[:10],
  }
)

preview

,u_idx,int_c_dv_no_proj,int_c_dv_proj_logit,int_c_dv_proj_identity,int_c_du_no_proj,int_c_du_proj_logit,int_c_du_proj_identity
0,0,0.560122,1.026600,1.033713,0.592060,1.0,1.0
1,1,0.793491,1.025172,1.032207,0.824993,1.0,1.0
2,2,0.970388,1.022999,1.030044,0.939329,1.0,1.0
3,3,1.042480,1.021129,1.027925,1.000439,1.0,1.0
4,4,1.063732,1.019266,1.025956,1.004521,1.0,1.0
5,5,1.062349,1.017504,1.024052,0.992975,1.0,1.0
6,6,1.061820,1.015995,1.022371,1.000843,1.0,1.0
7,7,1.060106,1.014605,1.020795,0.992205,1.0,1.0
8,8,1.076109,1.013331,1.019315,0.991069,1.0,1.0
9,9,1.051912,1.012077,1.017815,0.981363,1.0,1.0
